# Adaptive Noise Scheduling: Mathematical Derivation

This notebook documents the full objective and key invariants used in the implementation.

## 1. Forward Process with Adaptive Schedule

For each sample/class, ScheduleNet predicts a sequence $\{\beta_t(x)\}_{t=1}^T$ with bounds
\begin{align}
0 < \beta_{\min} < \beta_t(x) < \beta_{\max} \le 0.02.
\end{align}

Define
\begin{align}
\alpha_t(x) &= 1 - \beta_t(x), \\
\bar{\alpha}_t(x) &= \prod_{i=1}^{t} \alpha_i(x).
\end{align}

Then
\begin{align}
q(\mathbf{x}_t \mid \mathbf{x}_0, x)
= \mathcal{N}\left(\sqrt{\bar{\alpha}_t(x)}\,\mathbf{x}_0,
\left(1-\bar{\alpha}_t(x)\right)\mathbf{I}\right).
\end{align}

## 2. Reparameterized Sampling Equation

\begin{align}
\mathbf{x}_t = \sqrt{\bar{\alpha}_t(x)}\,\mathbf{x}_0 + \sqrt{1-\bar{\alpha}_t(x)}\,\boldsymbol{\epsilon}, \quad \boldsymbol{\epsilon}\sim\mathcal{N}(0,\mathbf{I}).
\end{align}

This is the equation implemented in `q_sample`.

## 3. Multi-objective Loss

### Diffusion loss
\begin{align}
\mathcal{L}_{\text{diffusion}} = \mathbb{E}\left[\lVert\boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t, y)\rVert_2^2\right].
\end{align}

### Efficiency regularizer
\begin{align}
\mathcal{L}_{\text{efficiency}} = \mathbb{E}_{x}\left[\sum_{t=1}^{T}\beta_t(x)\right].
\end{align}

### Smoothness regularizer
\begin{align}
\mathcal{L}_{\text{smoothness}} = \mathbb{E}_{x}\left[\sum_{t=1}^{T-1}\left(\beta_{t+1}(x)-\beta_t(x)\right)^2\right].
\end{align}

### Total objective
\begin{align}
\mathcal{L} = \mathcal{L}_{\text{diffusion}} + \lambda_1\mathcal{L}_{\text{efficiency}} + \lambda_2\mathcal{L}_{\text{smoothness}}.
\end{align}

## 4. Why Efficiency Encourages Fewer Reverse Steps

The expected corruption budget over a trajectory is proportional to $\sum_t \beta_t(x)$. Lower cumulative noise implies smaller expected denoising corrections per step, making accurate reconstruction achievable with fewer integration steps at fixed model capacity.

## 5. Invariants Enforced in Code

1. $\beta_{\min} < \beta_{\max}$ at all times.
2. $\bar{\alpha}_t$ strictly decreases with $t$.
3. For $T=1000$: $\bar{\alpha}_1 > 0.99$ and $\bar{\alpha}_T < 0.01$.
4. Output range check: $\beta_t(x)\in[0,0.02]$.

These are validated during forward calls and via dedicated tests.